# HP-4 · LoRA cho dịch máy đẳng thời (NLLB-200)

**Ý tưởng.** Dịch máy thường chỉ tối ưu *đúng nghĩa*. Lồng tiếng còn cần *đúng độ
dài*: câu tiếng Việt phải đọc vừa khung thời gian của câu gốc. Ở đây ta dạy
NLLB-200 nhận thêm một **token ngân sách** ở đầu câu nguồn — `<len:12>` nghĩa là
"hãy dịch sao cho đọc khoảng 12 âm tiết" — rồi fine-tune bằng LoRA.

**Vì sao LoRA.** NLLB-200 distilled 600M có ~600 triệu tham số. LoRA chỉ huấn
luyện vài triệu tham số thêm vào, vừa một GPU phổ thông, và giữ nguyên mô hình gốc.

**Trạng thái: CHƯA CHẠY.** Notebook này là đường ống hoàn chỉnh nhưng cần hai thứ
chưa có: một GPU và corpus song ngữ (đề cương HP-6: phụ đề OpenSubtitles). Các ô
huấn luyện có khoá `CHO_PHEP_CHAY` để không lỡ tay chạy hàng giờ.

**Nhóm A (huấn luyện).** Anh, Trung, Hàn, Nhật → Việt.
**Nhóm B (chỉ kiểm tra zero-shot).** Pháp, Nga, Thái → Việt. Đây là giả thuyết H1
của đề cương: ràng buộc độ dài học được có lan sang ngôn ngữ chưa huấn luyện không.

In [ ]:
# Chạy được dù mở notebook từ thư mục nào: đi ngược lên tới thư mục có backend/
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "backend").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Gốc dự án:", ROOT)

## Kiểm tra môi trường

In [ ]:
import importlib.util

need = {
    "torch": "--extra offline (hoặc --extra clone)",
    "transformers": "--extra offline",
    "peft": "uv pip install peft",
    "datasets": "uv pip install datasets",
}
missing = [n for n in need if importlib.util.find_spec(n) is None]

for name, how in need.items():
    mark = "✓" if name not in missing else "✗"
    print(f"  {mark} {name:<14} {'' if name not in missing else how}")

if missing:
    print("\nCòn thiếu:", ", ".join(missing))
else:
    import torch
    print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — LoRA sẽ rất chậm")

## Chuẩn bị dữ liệu

Mỗi mẫu gồm câu nguồn, câu đích tiếng Việt, và **ngân sách âm tiết** tính từ
chính câu đích trong tập huấn luyện. Lúc dùng thật thì ngân sách lấy từ khung
thời gian của cue chia cho tốc độ đọc (xem `backend/core/budget.py`).

In [ ]:
from backend.core import duration

BUDGET_TOKEN = "<len:{n}>"


def to_example(source_text: str, target_vi: str) -> dict:
    """Một mẫu huấn luyện: gắn token ngân sách lấy từ chính câu đích."""
    n = duration.syllables(target_vi)
    return {"src": f"{BUDGET_TOKEN.format(n=n)} {source_text}", "tgt": target_vi}


demo = [
    to_example("我們獲勝了", "Chúng ta đã thắng rồi"),
    to_example("I am the Cyclops and I will eat you all", "Ta là gã khổng lồ một mắt, ta sẽ ăn thịt các ngươi"),
]
for d in demo:
    print(d["src"])
    print("  ->", d["tgt"], f'({duration.syllables(d["tgt"])} âm tiết)')

In [ ]:
# Nạp corpus song ngữ thật. Định dạng mong đợi: JSONL mỗi dòng {"src": ..., "tgt": ...}
# HP-6 của đề cương sinh ra file này từ phụ đề OpenSubtitles.
CORPUS = ROOT / "data" / "corpus" / "vidub_train.jsonl"

if CORPUS.exists():
    import json

    pairs = [json.loads(ln) for ln in CORPUS.read_text(encoding="utf-8").splitlines() if ln.strip()]
    examples = [to_example(p["src"], p["tgt"]) for p in pairs]
    print(f"{len(examples)} mẫu từ {CORPUS}")
else:
    examples = demo
    print(f"Chưa có {CORPUS} — dùng tạm {len(demo)} mẫu minh hoạ.")
    print("HP-6 (dựng bộ dữ liệu ViDub) phải xong trước bước này.")

## Cấu hình LoRA

In [ ]:
BASE_MODEL = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR = ROOT / "data" / "models" / "nllb_lora_dangthoi"

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"], task_type="SEQ_2_SEQ_LM")
TRAIN = dict(learning_rate=2e-4, num_train_epochs=3,
             per_device_train_batch_size=8, gradient_accumulation_steps=2)

print("Mô hình gốc:", BASE_MODEL)
print("LoRA        :", LORA)
print("Huấn luyện  :", TRAIN)
print("Lưu vào     :", OUTPUT_DIR)

## Huấn luyện

⚠ Ô dưới đây chạy hàng giờ trên GPU. Đổi `CHO_PHEP_CHAY = True` khi đã có đủ
corpus và GPU.

In [ ]:
CHO_PHEP_CHAY = False   # đổi thành True để huấn luyện thật

if not CHO_PHEP_CHAY:
    print("Đang khoá. Đặt CHO_PHEP_CHAY = True để chạy thật.")
elif missing:
    print("Thiếu thư viện:", ", ".join(missing))
else:
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,
                              DataCollatorForSeq2Seq, Seq2SeqTrainer,
                              Seq2SeqTrainingArguments)

    tok = AutoTokenizer.from_pretrained(BASE_MODEL, src_lang="zho_Hans", tgt_lang="vie_Latn")
    # Token ngân sách là ký hiệu mới -> phải thêm vào từ điển rồi nới embedding
    tok.add_tokens([BUDGET_TOKEN.format(n=i) for i in range(1, 61)])

    base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
    base.resize_token_embeddings(len(tok))
    model = get_peft_model(base, LoraConfig(**LORA))
    model.print_trainable_parameters()

    def encode(batch):
        enc = tok(batch["src"], text_target=batch["tgt"], truncation=True, max_length=192)
        return enc

    ds = Dataset.from_list(examples).map(encode, batched=True, remove_columns=["src", "tgt"])

    trainer = Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(output_dir=str(OUTPUT_DIR), logging_steps=50,
                                      save_strategy="epoch", report_to=[], **TRAIN),
        train_dataset=ds,
        data_collator=DataCollatorForSeq2Seq(tok, model=model),
    )
    trainer.train()
    model.save_pretrained(OUTPUT_DIR)
    tok.save_pretrained(OUTPUT_DIR)
    print("Đã lưu ->", OUTPUT_DIR)

## Dịch thử với ngân sách khác nhau\n\nCùng một câu, ép ngân sách ngắn dần — câu dịch phải ngắn theo.

In [ ]:
def dich_voi_ngan_sach(text_nguon: str, n_am_tiet: int) -> str:
    """Dịch một câu, ép độ dài về khoảng `n_am_tiet` âm tiết."""
    if not OUTPUT_DIR.exists():
        return "(chưa có mô hình đã huấn luyện)"
    from peft import PeftModel
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
    base.resize_token_embeddings(len(tok))
    model = PeftModel.from_pretrained(base, OUTPUT_DIR)

    prompt = f"{BUDGET_TOKEN.format(n=n_am_tiet)} {text_nguon}"
    ids = tok(prompt, return_tensors="pt")
    out = model.generate(**ids, forced_bos_token_id=tok.convert_tokens_to_ids("vie_Latn"),
                         max_new_tokens=96)
    return tok.batch_decode(out, skip_special_tokens=True)[0]


for n in (6, 10, 16):
    print(f"<len:{n}> ->", dich_voi_ngan_sach("我是獨眼巨人，我要把你們都吃了", n))

## Đánh giá

Hai chỉ số đi cùng nhau, vì siết độ dài mà hỏng nghĩa thì vô nghĩa:

1. **BLEU** — nghĩa có giữ được không (`backend/core/metrics.py::bleu`).
2. **Tỉ lệ câu vượt khung** — đo bằng chính `budget.assess()` mà bản chạy thật dùng.

So ba cột: NLLB gốc · NLLB + LoRA đẳng thời · engine AI có vòng lặp rút gọn (HP-5).
Nhóm B (Pháp, Nga, Thái) chỉ chạy cột kiểm tra để trả lời giả thuyết H1.

In [ ]:
from backend.core import budget, metrics

print("BLEU:", "có sacrebleu" if metrics.bleu(["a"], ["a"]) is not None else "cần: uv pip install sacrebleu")
print("Ngưỡng vượt khung dùng chung:", budget.TOLERANCE, "giây")
print("Cách đo: budget.assess(cues, model) -> budget.over_budget(items)")